# dl_01_extract_procore

Pulls every endpoint in `Files/config/procore_endpoints.yml` into bronze.

Adding an endpoint is a YAML entry, not a change to this notebook.

**Rate limit:** Procore allows 600 requests/hour/client and does *not*
send `Retry-After` on a 429 - it sends `X-Rate-Limit-Reset`. The session
gates on the remaining-quota header before spending a request.

In [ ]:
import sys
sys.path.insert(0, "/lakehouse/default/Files/lib")

LIB = "/lakehouse/default/Files"

import requests

import fabric_common as fc
import scope as sc
import procore_extract as px
import watermark as wm
from ratelimit import RateLimitedSession, QuotaExhausted

CONFIG = f"{LIB}/config/procore_endpoints.yml"
batch_id = fc.new_batch_id()
print(f"batch {batch_id}")

In [ ]:
endpoints = sc.load_registry(CONFIG)
ordered = sc.resolution_order(endpoints)   # parents before their children
print(f"{len(ordered)} endpoints, resolution order:")
for endpoint in ordered:
    print(f"  {endpoint.name:32s} {endpoint.scope:8s} -> {endpoint.bronze_table}")

In [ ]:
settings = px.settings_from_secrets(fc.get_secret)
session = RateLimitedSession(requests.Session(), header_units="seconds")

token = px.fetch_token(settings, session)

# ACTIVE PROJECTS ONLY. Looping every project regardless of status is the
# fastest way to spend the hourly quota on jobs that closed three years ago.
projects = list(px.iter_active_projects(session, settings, token))
project_ids = [p["id"] for p in projects]
print(f"{len(project_ids)} active projects")

In [ ]:
fetched = {}
summary = []

for endpoint in ordered:
    parent_pairs = None
    if endpoint.parent:
        parent_pairs = sc.collect_parent_ids(
            fetched.get(endpoint.parent.endpoint, []), endpoint.parent
        )
        if not parent_pairs:
            # Not an error: a company with no prime contracts has no line items.
            print(f"  {endpoint.name:32s} skipped - parent "
                  f"{endpoint.parent.endpoint!r} returned nothing")
            summary.append((endpoint.name, 0, "skipped"))
            continue

    # Watermark is read BEFORE the pull and written only after it succeeds.
    since = wm.read_since(spark, endpoint.bronze_table, endpoint.name) if endpoint.incremental else None

    headers = px.build_headers(token, settings.company_id, endpoint)
    base_params = px.endpoint_params(endpoint, settings.company_id, since)

    records, rows = [], []
    ingested_at = fc.utc_now()
    try:
        for path, project_id in sc.expand_paths(
            endpoint, settings.company_id, project_ids, parent_pairs
        ):
            params = {**base_params, **px.implicit_params(endpoint, settings.company_id, project_id)}
            for record in px.iter_records(
                session, settings.base_url, path, headers, params=params, unwrap=endpoint.unwrap
            ):
                # Carry the project id onto the record so a child endpoint can
                # pair (parent_id, project_id) correctly.
                record.setdefault("_project_id", project_id)
                records.append(record)
                rows.append({
                    **px.to_bronze_row(record, endpoint, project_id, ingested_at),
                    "_batch_id": batch_id,
                    "_row_hash": fc.row_hash(record),
                })
    except QuotaExhausted as exc:
        # Stop cleanly rather than half-loading. Watermarks for endpoints already
        # done have advanced, so the next run resumes rather than restarting.
        print(f"  {endpoint.name:32s} QUOTA EXHAUSTED - {exc}")
        summary.append((endpoint.name, len(rows), "quota_exhausted"))
        fc.log_run(spark, batch_id, "extract_procore", endpoint.bronze_table,
                   len(rows), status="quota_exhausted", message=str(exc))
        break

    fetched[endpoint.name] = records

    if rows:
        df = spark.createDataFrame(rows)
        # MERGE on the composite key, not DROP + append: re-running is a no-op,
        # so the deliberate one-hour watermark overlap cannot duplicate rows.
        fc.merge_delta(spark, df, endpoint.bronze_table, ["_merge_key"])

        high = wm.high_water(records, "updated_at")
        if endpoint.incremental and high:
            wm.write_watermark(spark, endpoint.bronze_table, endpoint.name, high, batch_id)

    fc.log_run(spark, batch_id, "extract_procore", endpoint.bronze_table, len(rows))
    summary.append((endpoint.name, len(rows), "incremental" if since else "full"))
    print(f"  {endpoint.name:32s} {len(rows):7,d} rows  ({summary[-1][2]})")

In [ ]:
import json
import os


def write_diag(name: str, payload: dict) -> None:
    """Structured diagnostics to Files/_diag/.

    Fabric's job API gives no per-cell detail - a failed notebook reports
    "Failed" and nothing else. Writing what happened to a file the deploy
    scripts can read back is the difference between debugging this and guessing.
    """
    os.makedirs("/lakehouse/default/Files/_diag", exist_ok=True)
    path = f"/lakehouse/default/Files/_diag/{name}.json"
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=str)
    print(f"diagnostics -> {path}")

empty = [name for name, count, mode in summary if count == 0 and mode != "skipped"]
print(f"\nrequests made: {session.requests_made}  quota remaining: {session.remaining}")
if empty:
    # On a FULL reload an empty result usually means a permission gap or a tool
    # Data Link does not use - not genuinely zero records. Worth a look either way.
    print(f"empty endpoints ({len(empty)}): {', '.join(empty)}")

write_diag("extract_procore", {
    "batch_id": batch_id,
    "projects": len(project_ids),
    "requests": session.requests_made,
    "quota_remaining": session.remaining,
    "endpoints": [{"name": n, "rows": c, "mode": m} for n, c, m in summary],
})